# Story behind the project

In recent years, dating apps have become a common way for people to meet potential romantic partners. Many of these platforms use data science techniques to recommend matches, personalize user experiences, and identify patterns in user behavior. Because of that, dating-profile data offers an interesting opportunity to explore how personal attributes, preferences, and self-reported behaviors relate to one another.

In this portfolio project, I analyze a dataset of OKCupid profiles. OKCupid combines multiple-choice profile information with short written responses, which makes it useful for both structured machine learning tasks and natural language processing projects.

The purpose of this project is to practice formulating a machine learning question, preparing real-world data, training several classification models, and evaluating whether the available features contain useful predictive signal.

The data were provided by Codecademy.

LLMs was used to improve the commentary and provide feedback on what I could do better next time, but I did not implement those suggestions. In other words, I did not base my code on any generated code.

## Column descriptions

- **age:** continuous variable representing the user's age
- **body_type:** categorical variable describing the user's body type
- **diet:** categorical variable describing dietary preferences
- **drinks:** categorical variable describing alcohol consumption
- **drugs:** categorical variable describing drug usage
- **education:** categorical variable describing educational attainment
- **ethnicity:** categorical variable describing ethnic background
- **height:** continuous variable representing the user's height
- **income:** continuous variable representing reported income
- **job:** categorical variable describing employment
- **offspring:** categorical variable describing children status or preferences
- **orientation:** categorical variable describing sexual orientation
- **pets:** categorical variable describing pet preferences
- **religion:** categorical variable describing religious background
- **sex:** categorical variable describing gender
- **sign:** categorical variable describing astrological sign
- **smokes:** categorical variable describing smoking behavior
- **speaks:** categorical variable describing languages spoken
- **status:** categorical variable describing relationship status
- **last_online:** date variable showing the user's last login
- **location:** categorical variable describing user location

The dataset also includes open-ended short-answer responses:

- **essay0:** My self summary
- **essay1:** What I’m doing with my life
- **essay2:** I’m really good at
- **essay3:** The first thing people usually notice about me
- **essay4:** Favorite books, movies, shows, music, and food
- **essay5:** The six things I could never do without
- **essay6:** I spend a lot of time thinking about
- **essay7:** On a typical Friday night I am
- **essay8:** The most private thing I am willing to admit
- **essay9:** You should message me if…

For the scope of this project, I do not use the essay columns. These responses are text-heavy and would be better suited to a separate NLP-focused project.

## Project goal

The goal of this project is to predict a user's zodiac sign when that field is missing or not provided. Some users may consider zodiac signs relevant when choosing matches, so this project tests whether profile attributes contain enough information to predict that value.



In [2]:
import pandas as pd
import html
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("/home/grogy/python_playground/simple_ml_project/ml_projects/OKCupid-Date-A-Scientist-Starter/profiles.csv")

### Preprocessing

The first step is to prepare the data for machine learning. The target variable is `sign`, but many values include extra descriptive text. I clean this column by keeping only the first word, which represents the zodiac sign itself. I apply the same approach to `religion`, keeping only the broader religious category.



In [3]:
df["religion_clean"] = df.religion.str.split().str.get(0)
df["sign_clean"] = df.sign.str.split().str.get(0)
df = df.drop(columns=["sign","religion"])
df = df.loc[:,~df.columns.str.startswith("essay")]
df.head()

,age,body_type,diet,drinks,drugs,education,ethnicity,height,income,job,...,location,offspring,orientation,pets,sex,smokes,speaks,status,religion_clean,sign_clean
0,22,a little extra,strictly anything,socially,never,working on college/university,"asian, white",75.0,-1,transportation,...,"south san francisco, california","doesn&rsquo;t have kids, but might want them",straight,likes dogs and likes cats,m,sometimes,english,single,agnosticism,gemini
1,35,average,mostly other,often,sometimes,working on space camp,white,70.0,80000,hospitality / travel,...,"oakland, california","doesn&rsquo;t have kids, but might want them",straight,likes dogs and likes cats,m,no,"english (fluently), spanish (poorly), french (...",single,agnosticism,cancer
2,38,thin,anything,socially,NaN,graduated from masters program,NaN,68.0,-1,NaN,...,"san francisco, california",NaN,straight,has cats,m,no,"english, french, c++",available,NaN,pisces
3,23,thin,vegetarian,socially,NaN,working on college/university,white,71.0,20000,student,...,"berkeley, california",doesn&rsquo;t want kids,straight,likes cats,m,no,"english, german (poorly)",single,NaN,pisces
4,29,athletic,NaN,socially,never,graduated from college/university,"asian, black, other",66.0,-1,artistic / musical / writer,...,"san francisco, california",NaN,straight,likes dogs and likes cats,m,no,english,single,NaN,aquarius


Some columns contain data quality issues. In the `offspring` column, HTML entities such as `&rsquo;` appear instead of normal apostrophes. In the `income` column, missing values are represented as `-1` instead of `NaN`. I clean both issues before continuing.



In [4]:
df = df.map(lambda x: html.unescape(x) if isinstance(x, str) else x)
df.income = df.income.replace(-1, np.nan)
df.head()

,age,body_type,diet,drinks,drugs,education,ethnicity,height,income,job,...,location,offspring,orientation,pets,sex,smokes,speaks,status,religion_clean,sign_clean
0,22,a little extra,strictly anything,socially,never,working on college/university,"asian, white",75.0,NaN,transportation,...,"south san francisco, california","doesn’t have kids, but might want them",straight,likes dogs and likes cats,m,sometimes,english,single,agnosticism,gemini
1,35,average,mostly other,often,sometimes,working on space camp,white,70.0,80000.0,hospitality / travel,...,"oakland, california","doesn’t have kids, but might want them",straight,likes dogs and likes cats,m,no,"english (fluently), spanish (poorly), french (...",single,agnosticism,cancer
2,38,thin,anything,socially,NaN,graduated from masters program,NaN,68.0,NaN,NaN,...,"san francisco, california",NaN,straight,has cats,m,no,"english, french, c++",available,NaN,pisces
3,23,thin,vegetarian,socially,NaN,working on college/university,white,71.0,20000.0,student,...,"berkeley, california",doesn’t want kids,straight,likes cats,m,no,"english, german (poorly)",single,NaN,pisces
4,29,athletic,NaN,socially,never,graduated from college/university,"asian, black, other",66.0,NaN,artistic / musical / writer,...,"san francisco, california",NaN,straight,likes dogs and likes cats,m,no,english,single,NaN,aquarius


Next, I remove columns with more than 80% missing values. I also drop features that are unlikely to help predict zodiac sign or could introduce noise into the model.

Because zodiac signs are often associated with personality traits rather than demographic facts, I focus mainly on profile attributes related to preferences, lifestyle, and behavior.



In [5]:
df = df[df.columns[df.isna().mean() <= 0.8]]
df = df.drop(columns=["age","education","height", "ethnicity", "location", "speaks","last_online"])
df

,body_type,diet,drinks,drugs,job,offspring,orientation,pets,sex,smokes,status,religion_clean,sign_clean
0,a little extra,strictly anything,socially,never,transportation,"doesn’t have kids, but might want them",straight,likes dogs and likes cats,m,sometimes,single,agnosticism,gemini
1,average,mostly other,often,sometimes,hospitality / travel,"doesn’t have kids, but might want them",straight,likes dogs and likes cats,m,no,single,agnosticism,cancer
2,thin,anything,socially,NaN,NaN,NaN,straight,has cats,m,no,available,NaN,pisces
3,thin,vegetarian,socially,NaN,student,doesn’t want kids,straight,likes cats,m,no,single,NaN,pisces
4,athletic,NaN,socially,never,artistic / musical / writer,NaN,straight,likes dogs and likes cats,m,no,single,NaN,aquarius
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59941,NaN,NaN,socially,never,sales / marketing / biz dev,has kids,straight,has dogs,f,no,single,catholicism,cancer
59942,fit,mostly anything,often,sometimes,entertainment / media,doesn’t have kids,straight,likes dogs and likes cats,m,no,single,agnosticism,leo
59943,average,mostly anything,not at all,never,construction / craftsmanship,doesn’t have kids,straight,NaN,m,no,single,christianity,sagittarius
59944,athletic,mostly anything,socially,often,medicine / health,"doesn’t have kids, but wants them",straight,likes dogs and likes cats,m,trying to quit,single,agnosticism,leo


After removing the essay columns and selected low-value features, the dataset contains the structured variables that will be used for modeling.



Finally, I remove any remaining rows with missing values and one-hot encode the categorical columns. This converts categorical values into numeric indicator variables that can be used by scikit-learn models.



In [6]:
df = df.dropna()
for column in df.columns.to_list()[:-1]:
    df = pd.get_dummies(df,columns=[column], prefix=[column])

df.head()

,sign_clean,body_type_a little extra,body_type_athletic,body_type_average,body_type_curvy,body_type_fit,body_type_full figured,body_type_jacked,body_type_overweight,body_type_rather not say,...,status_single,religion_clean_agnosticism,religion_clean_atheism,religion_clean_buddhism,religion_clean_catholicism,religion_clean_christianity,religion_clean_hinduism,religion_clean_islam,religion_clean_judaism,religion_clean_other
0,gemini,True,False,False,False,False,False,False,False,False,...,True,True,False,False,False,False,False,False,False,False
1,cancer,False,False,True,False,False,False,False,False,False,...,True,True,False,False,False,False,False,False,False,False
7,sagittarius,False,False,True,False,False,False,False,False,False,...,True,False,False,False,False,True,False,False,False,False
14,taurus,False,False,False,False,False,False,False,False,False,...,True,False,False,False,True,False,False,False,False,False
19,pisces,False,True,False,False,False,False,False,False,False,...,True,False,False,False,True,False,False,False,False,False


## Target distribution and train-test split

Because the target variable has multiple classes, I first check the distribution of zodiac signs. If the classes are reasonably balanced, then accuracy is easier to interpret. If the classes are imbalanced, I need to compare model performance against a baseline rather than relying on accuracy alone.

After checking the target distribution, I split the data into training and test sets. I use stratified sampling so that each zodiac sign appears in similar proportions in both sets.



In [7]:
print(df.sign_clean.value_counts())
print(df.sign_clean.isna().sum())

sign_clean
gemini         719
leo            703
cancer         693
virgo          674
scorpio        638
sagittarius    627
libra          627
taurus         621
pisces         617
aries          612
capricorn      573
aquarius       560
Name: count, dtype: int64
0


In [8]:


Y = df["sign_clean"]
X = df.drop(columns=["sign_clean"])

test_ratio = 0.2
x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size=test_ratio, random_state=123, stratify=Y)


In [9]:
baseline = DummyClassifier(strategy="stratified").fit(x_train,y_train)
baseline_pred = baseline.predict(x_test)

In [10]:
print(classification_report(y_test, baseline_pred))

              precision    recall  f1-score   support

    aquarius       0.03      0.03      0.03       112
       aries       0.11      0.10      0.10       122
      cancer       0.10      0.09      0.10       139
   capricorn       0.09      0.10      0.10       115
      gemini       0.09      0.09      0.09       144
         leo       0.12      0.12      0.12       141
       libra       0.11      0.11      0.11       125
      pisces       0.07      0.07      0.07       123
 sagittarius       0.09      0.10      0.09       125
     scorpio       0.10      0.09      0.10       128
      taurus       0.13      0.13      0.13       124
       virgo       0.07      0.08      0.08       135

    accuracy                           0.09      1533
   macro avg       0.09      0.09      0.09      1533
weighted avg       0.09      0.09      0.09      1533



In [11]:
log_reg = LogisticRegression(max_iter=1000).fit(x_train, y_train)

log_reg_pred = log_reg.predict(x_test)


In [12]:
print(classification_report(y_test, log_reg_pred))

              precision    recall  f1-score   support

    aquarius       0.07      0.04      0.05       112
       aries       0.06      0.04      0.05       122
      cancer       0.08      0.08      0.08       139
   capricorn       0.07      0.03      0.05       115
      gemini       0.12      0.15      0.13       144
         leo       0.09      0.12      0.11       141
       libra       0.06      0.09      0.07       125
      pisces       0.07      0.07      0.07       123
 sagittarius       0.04      0.04      0.04       125
     scorpio       0.07      0.09      0.08       128
      taurus       0.11      0.09      0.10       124
       virgo       0.07      0.07      0.07       135

    accuracy                           0.08      1533
   macro avg       0.08      0.08      0.07      1533
weighted avg       0.08      0.08      0.08      1533



In [13]:
knn = KNeighborsClassifier().fit(x_train, y_train)
knn_pred = knn.predict(x_test)


In [14]:
print(classification_report(y_test, knn_pred))

              precision    recall  f1-score   support

    aquarius       0.07      0.19      0.10       112
       aries       0.10      0.20      0.13       122
      cancer       0.12      0.17      0.14       139
   capricorn       0.06      0.06      0.06       115
      gemini       0.07      0.06      0.06       144
         leo       0.05      0.04      0.05       141
       libra       0.16      0.10      0.13       125
      pisces       0.13      0.10      0.11       123
 sagittarius       0.11      0.06      0.08       125
     scorpio       0.11      0.05      0.07       128
      taurus       0.14      0.07      0.10       124
       virgo       0.08      0.04      0.05       135

    accuracy                           0.09      1533
   macro avg       0.10      0.10      0.09      1533
weighted avg       0.10      0.09      0.09      1533



In [15]:
dtree = DecisionTreeClassifier().fit(x_train,y_train)
dtree_predict = dtree.predict(x_test)

In [16]:
print(classification_report(y_test, dtree_predict))

              precision    recall  f1-score   support

    aquarius       0.06      0.07      0.06       112
       aries       0.06      0.06      0.06       122
      cancer       0.07      0.06      0.07       139
   capricorn       0.08      0.09      0.08       115
      gemini       0.08      0.08      0.08       144
         leo       0.09      0.08      0.08       141
       libra       0.07      0.08      0.08       125
      pisces       0.10      0.11      0.10       123
 sagittarius       0.05      0.05      0.05       125
     scorpio       0.12      0.11      0.12       128
      taurus       0.09      0.08      0.08       124
       virgo       0.11      0.13      0.12       135

    accuracy                           0.08      1533
   macro avg       0.08      0.08      0.08      1533
weighted avg       0.08      0.08      0.08      1533



## Results and interpretation

I tested whether structured dating-profile features can predict a user's zodiac sign. The models performed close to the baseline, which means they did not find meaningful predictive signal in the available profile attributes.

This result is useful: it suggests that the selected structured features do not contain enough information to reliably infer astrological sign. In practical terms, the model is mostly guessing. A stronger version of this project could test additional feature engineering, include the essay responses through NLP techniques, or evaluate whether a different target variable is more predictable from the same dataset.



In [17]:
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred)}")
print(f"Logistic regression accuracy: {accuracy_score(y_test, log_reg_pred)}")
print(f"KNN accuracy: {accuracy_score(y_test, knn_pred)}")
print(f"Decision tree accuracy: {accuracy_score(y_test, dtree_predict)}")

Baseline accuracy: 0.09328114807566862
Logistic regression accuracy: 0.07827788649706457
KNN accuracy: 0.09458577951728636
Decision tree accuracy: 0.08284409654272668
